In [1]:
1+1

2

In [2]:
import os
%pwd


'/workspaces/summarizer/textsummariser/research'

In [3]:
os.chdir("../")
%pwd

'/workspaces/summarizer/textsummariser'

## Basic Configuration

In [4]:
from dataclasses import dataclass
from pathlib import Path

@dataclass
class DataIngestionConfig:
    root_dir: Path
    source_URL: Path
    local_data_file:Path
    unzip_dir: Path

In [5]:
from src.textSummariser.constants import *
from src.textSummariser.utils.common import read_yaml, create_directories

## Configuration Updates

In [6]:
class ConfigurationManager:
    def __init__(self,
                config_path = CONFIG_FILE_PATH,
                params_path = PARAMS_FILE_PATH):
        self.config = read_yaml(config_path)
        self.params = read_yaml(params_path)

        create_directories([self.config.artifacts_root])

    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion
    
        create_directories([config.root_dir])

        data_ingestion_config = DataIngestionConfig(
            root_dir = Path(config.root_dir),
            source_URL = config.source_URL,
            local_data_file = Path(config.local_data_file),
            unzip_dir = Path(config.unzip_dir)
        )

        return data_ingestion_config

In [7]:
import os
import urllib.request as request
import zipfile
from src.textSummariser.logging import logger

## Components

In [8]:
class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config

    def download_file(self):
        if not os.path.exists(self.config.local_data_file):
            filename, headers = request.urlretrieve(
                url = str(self.config.source_URL),
                filename = str(self.config.local_data_file)  # Fixed the red underline!
            )
            logger.info(f"File downloaded! Info: \n{headers}")
        else:
            logger.info("File already exists")

    # Moved back one indentation level!
    def extract_zip_file(self):
        """
        zip_file_path: str
        Extracts the zip file into the data directory
        Function returns None
        """
        unzip_path = self.config.unzip_dir
        os.makedirs(unzip_path, exist_ok=True)
        with zipfile.ZipFile(self.config.local_data_file, "r") as zip_ref:
            zip_ref.extractall(unzip_path)
            logger.info(f"Extracted zip file to {unzip_path}")

In [9]:
config = ConfigurationManager()
data_ingestion_config = config.get_data_ingestion_config()
data_ingestion = DataIngestion(config=data_ingestion_config)

data_ingestion.download_file()
data_ingestion.extract_zip_file()

[2026-09-26 08:45:14,444: INFO: common: yaml file: config/config.yaml loaded successfully]
[2026-09-26 08:45:14,450: INFO: common: yaml file: params.yaml loaded successfully]
[2026-09-26 08:45:14,451: INFO: common: created directory at: artifacts]
[2026-09-26 08:45:14,452: INFO: common: created directory at: artifacts/data_ingestion]
[2026-09-26 08:45:15,430: INFO: 152520818: File downloaded! Info: 
Connection: close
Content-Length: 7903594
Cache-Control: max-age=300
Content-Security-Policy: default-src 'none'; style-src 'unsafe-inline'; sandbox
Content-Type: application/zip
ETag: "dbc016a060da18070593b83afff580c9b300f0b6ea4147a7988433e04df246ca"
Strict-Transport-Security: max-age=31536000
X-Content-Type-Options: nosniff
X-Frame-Options: deny
X-XSS-Protection: 1; mode=block
X-GitHub-Request-Id: 09BE:218C27:17BFC0:2F8A2B:6AB78618
x-github-edge-region: uksouth
Accept-Ranges: bytes
Date: Sat, 26 Sep 2026 08:45:15 GMT
Via: 1.1 varnish
X-Served-By: cache-lhr-egll1980073-LHR
X-Cache: MISS
X-